In [1]:
import pandas as pd
import psycopg2

In [2]:
# CSV 파일 불러오기
df = pd.read_csv("최종_카테고리DB.csv", encoding='utf-8')  # encoding 필요 시 수정

# 필요한 B열(CSV의 두 번째 열)과 C열(세 번째 열)만 가져오기
# 인덱스는 0부터 시작하므로 B=1, C=2
df = df.iloc[:, [1, 2]]  # B열, C열
df.columns = ['code', 'name']  # 컬럼명 설정

df.head()  # 데이터 확인

,code,name
0,spa_0001,거실
1,spa_0002,다이닝룸
2,spa_0003,드레스룸
3,spa_0004,발코니
4,spa_0005,서재/멀티룸


In [ ]:
import os
from dotenv import load_dotenv

DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")

try:
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )

    # 커서 생성
    cur = conn.cursor()
    cur.execute("SELECT version();")
    print(cur.fetchone())

except Exception as e:
    print("DB 연결 에러 : ", e)
finally:
    if conn:
        conn.close()

In [ ]:
# 뉴스, 블로그 데이터 넣기

import pandas as pd
import psycopg2

# 1. CSV 불러오기
df = pd.read_csv("hanssem_contents.csv", encoding='utf-8')

# 2. pubdate를 datetime으로 변환 (필요시)
df['pubdate'] = pd.to_datetime(df['pubdate'], errors='coerce')

# 3. pubdate 기준 정렬
df = df.sort_values(by='pubdate').reset_index(drop=True)

# 4. contents_id 부여 (1부터 시작)
df['contents_id'] = df.index + 1

# 5. 컬럼 순서 맞추기
df = df[['contents_id', 'title', 'url', 'pubdate', 'source']]

# 6. DB 연결 정보
conn = psycopg2.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT
)
cur = conn.cursor()

# 7. INSERT
for _, row in df.iterrows():
    cur.execute("""
        INSERT INTO contents (contents_id, title, url, pubdate, source)
        VALUES (%s, %s, %s, %s, %s)
    """, (row['contents_id'], row['title'], row['url'], row['pubdate'], row['source']))

# 8. 마무리
conn.commit()
cur.close()
conn.close()


In [ ]:
# 시공사례 데이터 넣기

import pandas as pd
import psycopg2

# CSV 불러오기
df = pd.read_csv("master_info_eda_final.csv", encoding='utf-8')

# 컬럼 매핑: DataFrame 컬럼명을 테이블 컬럼에 맞게 정리
df = df.rename(columns={
    'Seq': 'built_case_id',
    'Title': 'title',
    'apply_cost': 'cost',
    'apply_style': 'style',
    'apply_space': 'apply_space',
    'TagList': 'taglist'
})

# 필요 시, null/결측값 처리
df = df.fillna('')  # 또는 적절한 처리

# DB 연결
conn = psycopg2.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT
)
cur = conn.cursor()

# INSERT 반복
for _, row in df.iterrows():
    cur.execute("""
        INSERT INTO Built_case (built_case_id, title, cost, style, apply_space, taglist)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (
        row['built_case_id'],
        row['title'],
        row['cost'],
        row['style'],
        row['apply_space'],
        row['taglist']
    ))

# 커밋 및 종료
conn.commit()
cur.close()
conn.close()

In [ ]:
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT
)

df = pd.read_sql("SELECT * FROM built_case", conn)
print(df)
conn.close()


      built_case_id                                  title      cost  \
0             11108        광진구 구의현대2단지 타워형 빗각주방의 정석을 확인하세요  cos_0009   
1             10144                     확장없이도 넓어보이는 아파트 살기  cos_0009   
2             14571                            3대가 함께 사는 집  cos_0002   
3             11232       목동신시가지14단지 28평 중문과 주방에 포인트를 준 공간  cos_0005   
4             11236     영등포구 대림갑을명가 31평형 레이아웃 변경된 화이트 인테리어  cos_0009   
...             ...                                    ...       ...   
3472           9982                도심 속 아파트에서 실현하는 프렌치 라이프  cos_0007   
3473           9984                     새 가족을 기다리는 실속 인테리어  cos_0005   
3474           9986                    여유로운 육아 라이프가 실현되는 집  cos_0005   
3475           9989                      정서발달을 위한 아이방 인테리어  cos_0005   
3476           9993  거실 속 고즈넉한 툇마루 화이트톤의 미니멀 50평대 아파트 리모델링  cos_0009   

         style                                        apply_space  \
0     sty_0002  spa_0008,spa_0001,spa_0009,spa_0010,spa_0007,s... 

/tmp/ipykernel_200329/1268901456.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM built_case", conn)


In [ ]:
## 카테고리 정보 넣기

# CSV 읽기
df = pd.read_csv("최종_카테고리DB.csv", encoding='utf-8').iloc[:, [1, 2]]
df.columns = ['code', 'name']

# DB 연결
conn = psycopg2.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT
)
cur = conn.cursor()

# INSERT 반복
for _, row in df.iterrows():
    cur.execute(
        "INSERT INTO category (code, name) VALUES (%s, %s)",
        (row['code'], row['name'])
    )

# 마무리
conn.commit()
cur.close()
conn.close()


UniqueViolation: duplicate key value violates unique constraint "category_pkey"
DETAIL:  Key (code)=(fur_0000) already exists.
